# SaludAI — Agent Queries

This notebook demonstrates how to use the SaludAI agent to answer clinical questions in natural language.

The agent:
1. Interprets the user's question
2. Resolves clinical terminology (SNOMED CT, ICD-10, LOINC, ATC)
3. Builds and executes FHIR queries
4. Analyzes results with sandboxed Python code
5. Generates a narrative answer with sources

## Prerequisites

```bash
docker compose up -d          # HAPI FHIR with synthetic data
cp .env.example .env          # Configure API keys (Anthropic/OpenAI)
```

## 1. Configure the Agent

The agent needs an LLM (Anthropic, OpenAI, or Ollama), a FHIR client, and a locale pack.

**Locale packs** adapt the agent to different health systems. Choose `"ar"` for Argentina (Spanish) or `"en_us"` for United States (English).

In [2]:
from dotenv import load_dotenv

load_dotenv()

from saludai_agent.config import AgentConfig
from saludai_agent.llm import create_llm_client
from saludai_agent.loop import AgentLoop
from saludai_agent.tracing import create_tracer
from saludai_core.fhir_client import FHIRClient
from saludai_core.locales import load_locale_pack
from saludai_core.terminology import TerminologyResolver

# --- Choose your locale ---
LOCALE = "en_us"  # Options: "ar" (Argentina/Spanish), "en_us" (US/English)

agent_config = AgentConfig()
locale_pack = load_locale_pack(LOCALE)

print(f"Locale: {locale_pack.name} ({locale_pack.language})")
print(f"LLM: {agent_config.llm_provider}/{agent_config.llm_model}")
print(f"Max iterations: {agent_config.agent_max_iterations}")
print(f"Terminology systems: {[s.key for s in locale_pack.terminology_systems]}")

Locale: United States (en)
LLM: anthropic/claude-sonnet-4-5-20250929
Max iterations: 8
Terminology systems: ['snomed_ct', 'icd_10_cm', 'loinc', 'atc']


## 2. Query Helper

A helper function that initializes the agent and runs a query, displaying results in a readable format.

In [3]:
async def ask(query: str) -> None:
    """Run a query through the agent and display results."""
    llm = create_llm_client(agent_config)
    resolver = TerminologyResolver(locale_pack=locale_pack)
    tracer = create_tracer(agent_config)

    async with FHIRClient() as client:
        loop = AgentLoop(
            llm=llm,
            fhir_client=client,
            terminology_resolver=resolver,
            config=agent_config,
            tracer=tracer,
            locale_pack=locale_pack,
        )
        result = await loop.run(query)

    print(f"Question: {result.query}")
    print(f"{'=' * 60}")
    print(f"\nAnswer:\n{result.answer}")
    print(f"\n{'=' * 60}")
    print(f"Iterations: {result.iterations}")
    print(f"Tools used: {len(result.tool_calls_made)}")
    for tc in result.tool_calls_made:
        print(f"  - {tc.name}({tc.arguments})")
    if result.trace_url:
        print(f"\nLangfuse trace: {result.trace_url}")

## 3. Example Queries

### Simple: Count (FHIR-native, uses `_summary=count`)

In [ ]:
# English
await ask("How many patients with type 2 diabetes are there?")

# Spanish — uncomment to try:
# await ask("¿Cuántos pacientes con diabetes tipo 2 hay?")

### Per-patient: Conditions for a specific patient

In [ ]:
# English
await ask("What conditions has patient 1005 been diagnosed with?")

# Spanish — uncomment to try:
# await ask("¿Qué condiciones tiene diagnosticadas el paciente 1005?")

### Per-patient: Multi-resource navigation

These queries require the agent to traverse references across FHIR resources for a single patient.

In [ ]:
# English
await ask("What medications is patient 1005 currently taking?")

# Spanish — uncomment to try:
# await ask("¿Qué medicamentos toma actualmente el paciente 1005?")

In [ ]:
# English
await ask("Does patient 1010 have any allergies? If so, list them.")

# Spanish — uncomment to try:
# await ask("¿El paciente 1010 tiene alguna alergia? Si es así, listarlas.")

### FHIR-native: Cross-resource count (uses `_has`, no data transfer)

In [ ]:
# English
await ask("How many women over 50 have been diagnosed with hypertension?")

# Spanish — uncomment to try:
# await ask("¿Cuántas mujeres mayores de 50 tienen hipertensión?")

## 4. Try Your Own Query

In [ ]:
await ask("What was the last encounter for patient 1020, and what was its type?")

---

**Next:** In [notebook 03](03-benchmark-eval.ipynb) we evaluate the agent's accuracy with the FHIR-AgentBench benchmark.